In [1]:
print(1,2,3)

1 2 3


In [2]:
from pathlib import Path
from urllib.request import urlretrieve

PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

files = {
    "ingest.py": f"{PREFIX}/01-agentic-rag/code/ingest.py",
    "rag_helper.py": f"{PREFIX}/01-agentic-rag/code/rag_helper.py",
    "evaluation_utils.py": f"{PREFIX}/04-evaluation/code/evaluation_utils.py",
}

for filename, url in files.items():
    urlretrieve(url, filename)
    print(f"Heruntergeladen: {filename}")

Heruntergeladen: ingest.py
Heruntergeladen: rag_helper.py
Heruntergeladen: evaluation_utils.py


In [3]:
from ingest import load_faq_data
from evaluation_utils import llm_structured, calc_price

In [4]:
from ingest import load_faq_data
documents = load_faq_data()

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [6]:
documents = documents_llm

In [7]:
doc = documents[0]
doc["doc_id"] = doc.pop("id")
print(doc["question"])
print(doc["answer"])

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.", "doc_id": "74eb249bbf"}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
response.output_parsed.questions


['I just found this course — am I still allowed to enroll, or is it too late?',
 'Can I join the course after it has already started?',
 'If I start now, is there still a way to get the certificate?',
 'What do I need to do to qualify for the certificate if I join late?',
 'Are project submissions still open for new students who discovered the course recently?']

In [16]:
doc

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [17]:
len(documents)

118

In [18]:
from evaluation_utils import llm_structured

In [19]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I found it late?', 'If I join now, do I still have a shot at getting a certificate?', 'Is it okay to start the course after it already began?', 'What do I need to do to be eligible for the certificate?', 'Are project submissions still open for new students?']


In [20]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=78, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=285)

In [21]:
from evaluation_utils import calc_price

In [22]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00035099999999999997,
 'total_cost': 0.00050625}

In [23]:
records = []

for q in result.questions:
    records = []

    for q in result.questions:
        records.append({
            "question": q,
            "document": doc.get("doc_id", doc.get("id"))
        })

    records

records

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, do I still have a shot at getting a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course after it already began?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Are project submissions still open for new students?',
  'document': '74eb249bbf'}]

In [24]:
import pandas as pd

In [25]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,"If I join now, do I still have a shot at getti...",74eb249bbf
2,Is it okay to start the course after it alread...,74eb249bbf
3,What do I need to do to be eligible for the ce...,74eb249bbf
4,Are project submissions still open for new stu...,74eb249bbf


### Generating Ground Truth for All Documents

In [26]:
from evaluation_utils import llm_structured_retry

In [27]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [28]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []
    doc_id = doc.get("doc_id", doc.get("id"))

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc_id
        })

    return results, usage

In [29]:
generate_ground_truth(doc)

([{'question': 'Can I still start the course now if I just found out about it?',
   'document': '74eb249bbf'},
  {'question': 'Is it too late to join the course after it already started?',
   'document': '74eb249bbf'},
  {'question': 'If I join late, can I still get a certificate somehow?',
   'document': '74eb249bbf'},
  {'question': 'Do I need to submit the project while submissions are still open to be eligible for the certificate?',
   'document': '74eb249bbf'},
  {'question': 'What’s the rule for getting the course certificate if I’m joining after the course has begun?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=95, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=302))

In [30]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    def generate_ground_truth(doc):
        user_prompt = json.dumps(doc)

        out, usage = llm_structured_retry(
            openai_client,
            data_gen_instructions,
            user_prompt,
            Questions
        )

        results = []
        doc_id = doc.get("doc_id", doc.get("id"))

        for q in out.questions:
            results.append({
                "question": q,
                "document": doc_id
            })

        return results, usage
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

Parallel processing

Running the calls one after another wastes most of the time waiting on the network. Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. We process the documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

Import ThreadPoolExecutor:

In [31]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [32]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/118 [00:00<?, ?it/s]

In [33]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

590

In [34]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop if the Zoom link isn’t shared with us?',
 'document': '489dd1c9d9'}

Calculate the total cost:



In [35]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.09064200000000003

In [36]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.09064200000000003

In [37]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [38]:
df_ground_truth

,question,document
0,Can I still start the course if I just found i...,74eb249bbf
1,Is it too late to join the course or can I sti...,74eb249bbf
2,"If I begin the course now, am I still eligible...",74eb249bbf
3,What do I need to do to get the certificate if...,74eb249bbf
4,Do I have to submit my project before submissi...,74eb249bbf
...,...,...
585,My EU Logfire token keeps getting 401 Unauthor...,46efd1088d
586,Do I need to set the Logfire EU base URL manua...,46efd1088d
587,Could a wrong `.env` setup make Logfire reject...,46efd1088d
588,What's the difference between `LOGFIRE_TOKEN` ...,46efd1088d


In [39]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [40]:
len(df_ground_truth)

590